# Analysis of how geopolitical crises affect certain commodities
# How has Guardian coverage of commodities evolved in response to geopolitical crises, and what patterns emerge in the relationship between media attention and commodity price dynamics?

# Introduction:

*Stuff to add*

- Contents structure
- Explain what the project is about
- How we carried out the analysis
- What are the tools used



# 

## Data Acquisition:

### Primary: Guardian API
We will collect articles from the Guardian API using commodity-related search terms (e.g. oil, natural gas, copper, wheat, OPEC, energy crisis) filtered by relevant sections (e.g. Business, Environment, World News). For each article, we may extract the publication date, headline, section, tags, word count, and body text. We plan to collect data spanning approximately 2014 - 2026 to capture multiple geopolitical cycles.

### Secondary: Commodity Price Data
To contextualise media coverage against market movements, we will source historical commodity price data from Yahoo Finance (via the yfinance Python library) or publicly available datasets. This will include daily/weekly prices for key commodities such as Brent Crude, natural gas, gold, copper, and wheat.

- Query the Guardian API with commodity-related keywords and display results across the full date range

- Download commodity price time series from Yahoo Finance using yfinance

- Store raw data in structured formats (JSON/CSV) for reproducibility



In [1]:
import json
import requests
import pandas as pd
import time
from datetime import date

with open('keys.json') as f:
    key = json.load(f)

API_KEY = key['guardian']['api_key']
BASE_URL = 'https://content.guardianapis.com'


date_ranges = [
    ("2020-01-01", "2020-06-30"),
    ("2020-07-01", "2020-12-31"),
    ("2021-01-01", "2021-06-30"),
    ("2021-07-01", "2021-12-31"),
    ("2022-01-01", "2022-06-30"),
    ("2022-07-01", "2022-12-31"),
    ("2023-01-01", "2023-06-30"),
    ("2023-07-01", "2023-12-31"),
    ("2024-01-01", "2024-06-30"),
    ("2024-07-01", "2024-12-31"),
    ("2025-01-01", "2025-12-31"),
]

all_results = []

for from_date, to_date in date_ranges:
    for page in range(1, 11): #21
        parameters = {
            "api-key": API_KEY,
            "q": "oil OR natural gas OR gold OR OPEC OR energy crisis",
            "page-size": 50,
            "page": page,
            "show-fields": "bodyText",
            "from-date": from_date,
            "to-date": to_date,
            "order-by": "oldest"
        }

        response = requests.get(f"{BASE_URL}/search", params=parameters)
        response.raise_for_status()
        data = response.json()['response']


        results = data['results']
        total_pages = data['pages']

        for article in results:
            all_results.append({
                'date': article.get('webPublicationDate'),
                'section': article.get('sectionName'),
                'title': article.get('webTitle'),
                'body': article.get('fields', {}).get('bodyText', '')
            })

        if page >= total_pages:
            break

        time.sleep(0.1)

    print(f"  Total so far: {len(all_results)} articles")

df = pd.DataFrame(all_results)
df['date'] = pd.to_datetime(df['date'])
df = df.drop_duplicates(subset=['title'])
df.to_csv('data/guardian_commodities.csv', index=False)
print(f"\nFinal: {len(df)} articles saved")



  Total so far: 500 articles
  Total so far: 1000 articles
  Total so far: 1500 articles
  Total so far: 2000 articles
  Total so far: 2500 articles
  Total so far: 3000 articles
  Total so far: 3500 articles
  Total so far: 4000 articles
  Total so far: 4500 articles
  Total so far: 5000 articles
  Total so far: 5500 articles

Final: 5495 articles saved


In [3]:
import yfinance as yf


# Define commodity tickers
# Yahoo Finance uses these symbols for common commodities:
# all futures
commodities = {
    "Gold":        "GC=F",   
    "Brent Oil":   "BZ=F",   # brent crude oil
    "Natural Gas": "NG=F",   
    #"Wheat":       "ZW=F",   
}

# Download historical data
data = yf.download(
    tickers=list(commodities.values()),
    start="2020-01-01",
    end="2025-12-31",
    interval="1wk"       # 1d for daily, 1wk for weekly, 1mo for monthly
)

# Extract closing prices only
close_prices = data["Close"]
close_prices.columns = list(commodities.keys())

df_commodities_prices = pd.DataFrame(close_prices)


print(close_prices.head(20))


[*********************100%***********************]  3 of 3 completed

                 Gold    Brent Oil  Natural Gas
Date                                           
2020-01-01  68.269997  1571.800049        2.162
2020-01-08  64.489998  1542.400024        2.187
2020-01-15  64.589996  1556.400024        1.895
2020-01-22  59.509998  1569.199951        1.934
2020-01-29  53.959999  1550.400024        1.872
2020-02-05  54.009998  1565.599976        1.788
2020-02-12  57.750000  1600.000000        1.981
2020-02-19  54.950001  1646.900024        1.847
2020-02-26  51.860001  1642.099976        1.800
2020-03-04  37.220001  1659.099976        1.936
2020-03-11  28.730000  1524.900024        1.729
2020-03-18  27.150000  1660.199951        1.653
2020-03-25  22.740000  1583.400024        1.640
2020-04-01  31.870001  1664.800049        1.852
2020-04-08  29.600000  1756.699951        1.650
2020-04-15  19.330000  1678.199951        1.821
2020-04-22  20.459999  1710.500000        1.794
2020-04-29  30.969999  1704.400024        2.134
2020-05-06  29.980000  1704.400024      

# IDA

In [ ]:
# mantra 

# Data wrangling

# EDA  

# Visualisation

rian